**Experimenting with Different Basis Functions**

Comparing polynomial, Fourier and Gaussian RBF basis functions using the same data split.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.loadtxt("data/noisy_2.txt")
x = data[:, 0]
y = data[:, 1]

np.random.seed(42)
indices = np.random.permutation(len(x))
x_shuffled = x[indices]
y_shuffled = y[indices]

train_end = int(0.6 * len(x))
val_end = int(0.8 * len(x))

x_train = x_shuffled[:train_end]
y_train = y_shuffled[:train_end]
x_val = x_shuffled[train_end:val_end]
y_val = y_shuffled[train_end:val_end]
x_test = x_shuffled[val_end:]
y_test = y_shuffled[val_end:]

In [ ]:
def least_squares(X, y):
    A = X.T @ X
    b = X.T @ y
    return np.linalg.inv(A) @ b

def mse(y_actual, y_pred):
    return np.mean((y_actual - y_pred) ** 2)

def polynomial_basis(x, degree, x_mean, x_std):
    z = (x - x_mean) / x_std
    X = np.ones((len(x), degree + 1))
    for power in range(1, degree + 1):
        X[:, power] = z ** power
    return X

def fourier_basis(x, terms, x_min, x_max):
    angle = 2 * np.pi * (x - x_min) / (x_max - x_min)
    X = np.ones((len(x), 2 * terms + 1))
    for term in range(1, terms + 1):
        X[:, 2 * term - 1] = np.sin(term * angle)
        X[:, 2 * term] = np.cos(term * angle)
    return X

def rbf_basis(x, count, x_min, x_max):
    centers = np.linspace(x_min, x_max, count)
    width = 1.5 * (centers[1] - centers[0])
    X = np.ones((len(x), count + 1))
    for index, center in enumerate(centers):
        X[:, index + 1] = np.exp(-0.5 * ((x - center) / width) ** 2)
    return X

In [ ]:
# selecting the number of Fourier terms using validation MSE

x_min = np.min(x_train)
x_max = np.max(x_train)
fourier_terms = range(1, 16)
fourier_val_errors = []

for terms in fourier_terms:
    X_train = fourier_basis(x_train, terms, x_min, x_max)
    X_val = fourier_basis(x_val, terms, x_min, x_max)
    weights = least_squares(X_train, y_train)
    fourier_val_errors.append(mse(y_val, X_val @ weights))

best_fourier_terms = fourier_terms[np.argmin(fourier_val_errors)]
print(f"Best Fourier terms: {best_fourier_terms}")
print(f"Fourier validation MSE: {min(fourier_val_errors):.4f}")

In [ ]:
# selecting the number of RBF centers using validation MSE

rbf_counts = [5, 7, 9, 11, 15, 20, 25, 30, 40]
rbf_val_errors = []

for count in rbf_counts:
    X_train = rbf_basis(x_train, count, x_min, x_max)
    X_val = rbf_basis(x_val, count, x_min, x_max)
    weights = least_squares(X_train, y_train)
    rbf_val_errors.append(mse(y_val, X_val @ weights))

best_rbf_count = rbf_counts[np.argmin(rbf_val_errors)]
print(f"Best RBF centers: {best_rbf_count}")
print(f"RBF validation MSE: {min(rbf_val_errors):.4f}")

In [ ]:
# comparing the fitted curves

x_mean = np.mean(x_train)
x_std = np.std(x_train)
x_curve = np.linspace(np.min(x), np.max(x), 1000)

X_train_poly = polynomial_basis(x_train, 10, x_mean, x_std)
poly_weights = least_squares(X_train_poly, y_train)
poly_curve = polynomial_basis(x_curve, 10, x_mean, x_std) @ poly_weights
poly_val_mse = mse(y_val, polynomial_basis(x_val, 10, x_mean, x_std) @ poly_weights)

X_train_fourier = fourier_basis(x_train, best_fourier_terms, x_min, x_max)
fourier_weights = least_squares(X_train_fourier, y_train)
fourier_curve = fourier_basis(x_curve, best_fourier_terms, x_min, x_max) @ fourier_weights

X_train_rbf = rbf_basis(x_train, best_rbf_count, x_min, x_max)
rbf_weights = least_squares(X_train_rbf, y_train)
rbf_curve = rbf_basis(x_curve, best_rbf_count, x_min, x_max) @ rbf_weights

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
curves = [poly_curve, fourier_curve, rbf_curve]
titles = [
    f"Polynomial degree 10 - Val MSE: {poly_val_mse:.2f}",
    f"Fourier {best_fourier_terms} terms - Val MSE: {min(fourier_val_errors):.2f}",
    f"RBF {best_rbf_count} centers - Val MSE: {min(rbf_val_errors):.2f}",
]

for axis, curve, title in zip(axes, curves, titles):
    axis.scatter(x, y, s=0.5)
    axis.plot(x_curve, curve, color="red", linewidth=2)
    axis.set_title(title)
    axis.set_xlabel("x")
    axis.set_ylabel("y")

plt.tight_layout()
plt.savefig("../docs/lab3/submissions/figures/basis_function_comparison.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

In [ ]:
# refitting each selected model on train + validation data

x_final = np.concatenate((x_train, x_val))
y_final = np.concatenate((y_train, y_val))

x_mean_final = np.mean(x_final)
x_std_final = np.std(x_final)
X_final = polynomial_basis(x_final, 10, x_mean_final, x_std_final)
weights = least_squares(X_final, y_final)
poly_test_mse = mse(y_test, polynomial_basis(x_test, 10, x_mean_final, x_std_final) @ weights)

x_min_final = np.min(x_final)
x_max_final = np.max(x_final)
X_final = fourier_basis(x_final, best_fourier_terms, x_min_final, x_max_final)
weights = least_squares(X_final, y_final)
fourier_test_mse = mse(y_test, fourier_basis(x_test, best_fourier_terms, x_min_final, x_max_final) @ weights)

X_final = rbf_basis(x_final, best_rbf_count, x_min_final, x_max_final)
weights = least_squares(X_final, y_final)
rbf_test_mse = mse(y_test, rbf_basis(x_test, best_rbf_count, x_min_final, x_max_final) @ weights)

print(f"Polynomial test MSE: {poly_test_mse:.4f}")
print(f"Fourier test MSE: {fourier_test_mse:.4f}")
print(f"RBF test MSE: {rbf_test_mse:.4f}")